In [1]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from src.data import load_data, clean_data, split_data
from src.features import build_preprocessor

df = clean_data(load_data("../data/telco_churn.csv"))
X_train, X_test, y_train, y_test = split_data(df)

# XGBoost's way of handling imbalance: weight churners by (stayers / churners)
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("Class ratio:", round(ratio, 2))

Class ratio: 2.77


In [2]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5,
        class_weight="balanced_subsample", random_state=42, n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=3,
        scale_pos_weight=ratio, random_state=42, eval_metric="logloss",
    ),
}

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision"}

rows = []
for name, clf in models.items():
    pipe = Pipeline([("prep", build_preprocessor(X_train)), ("clf", clf)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    rows.append({
        "model": name,
        "ROC-AUC": scores["test_roc_auc"].mean(),
        "ROC-AUC std": scores["test_roc_auc"].std(),
        "PR-AUC": scores["test_pr_auc"].mean(),
        "PR-AUC std": scores["test_pr_auc"].std(),
    })

results = pd.DataFrame(rows).round(3)
results

,model,ROC-AUC,ROC-AUC std,PR-AUC,PR-AUC std
0,Logistic Regression,0.846,0.012,0.660,0.019
1,Random Forest,0.845,0.009,0.653,0.020
2,XGBoost,0.846,0.011,0.667,0.022
